In [2]:
"""
Muestrea tweets REALES del corpus para cada palabra de control (monosémica/
polisémica), para validar manualmente si esa etiqueta realmente corresponde
al uso en ESTE corpus -- no basta con conocimiento general del idioma (ver
caso "elefante": animal / insulto / "elefante en la habitación", puede que
solo uno de esos sentidos aparezca en los tweets reales, o puede haber usos
coloquiales que el diccionario no captura).

Usa los tweets ya deduplicados (un representante por grupo de casi-duplicado)
para no sesgar la muestra con spam repetido.

Uso:
    python muestrear_control.py
"""

import re
import random
import pandas as pd

TEXTOS_CSV = "output/all_tweets.csv"                    # columnas: id_tweet, texto
GRUPOS_DUPLICADOS_CSV = "output/all_tweets_grupos_duplicados.csv"  # columnas: id_tweet, es_representante
N_MUESTRA_POR_PALABRA = 15
SEMILLA = 42

PALABRAS_CALIBRACION_POLISEMICAS = {"banco", "gato", "planta", "sierra", "cola", "papa", "vela", "muñeca"}
PALABRAS_CALIBRACION_MONOSEMICAS = {"bicicleta", "computadora", "cumpleaños", "desayuno",
                                      "aeropuerto", "estudiante", "diciembre", "elefante"}


def encontrar_ocurrencias_regex(palabra, texto):
    patron = r"\b" + re.escape(palabra) + r"\b"
    return list(re.finditer(patron, texto, flags=re.IGNORECASE))


def main():
    textos_df = pd.read_csv(TEXTOS_CSV)
    grupos_df = pd.read_csv(GRUPOS_DUPLICADOS_CSV)
    representantes = set(grupos_df.loc[grupos_df["es_representante"], "id_tweet"])
    textos_df = textos_df[textos_df["id_tweet"].isin(representantes)]

    rng = random.Random(SEMILLA)

    palabras = sorted(PALABRAS_CALIBRACION_POLISEMICAS | PALABRAS_CALIBRACION_MONOSEMICAS)
    reporte = []

    for palabra in palabras:
        tipo_asumido = "polisémica" if palabra in PALABRAS_CALIBRACION_POLISEMICAS else "monosémica"
        coincidencias = []
        for _, row in textos_df.iterrows():
            texto = str(row["texto"])
            if encontrar_ocurrencias_regex(palabra, texto):
                coincidencias.append((row["id_tweet"], texto))

        muestra = rng.sample(coincidencias, min(N_MUESTRA_POR_PALABRA, len(coincidencias)))

        print(f"\n{'='*70}")
        print(f"PALABRA: '{palabra}'  (etiqueta asumida: {tipo_asumido})  "
              f"-- {len(coincidencias)} tweets totales, muestra de {len(muestra)}")
        print("=" * 70)
        for id_tweet, texto in muestra:
            print(f"[{id_tweet}] {texto}")

        reporte.append({"palabra": palabra, "tipo_asumido": tipo_asumido, "n_total_ocurrencias": len(coincidencias)})

    pd.DataFrame(reporte).to_csv("control_muestra_resumen.csv", index=False)
    print("\n\nResumen guardado en control_muestra_resumen.csv")
    print("\nRevisa cada bloque y anota: ¿cuántos sentidos DISTINTOS ves realmente en estos tweets?")
    print("Si una 'monosémica' muestra >=2 sentidos genuinos, o una 'polisémica' muestra solo 1,")
    print("hay que corregir su etiqueta antes de recalcular el Youden J.")


if __name__ == "__main__":
    main()


PALABRA: 'aeropuerto'  (etiqueta asumida: monosémica)  -- 128 tweets totales, muestra de 15
[12009] Mexico_City _GEO Que ganas de estar en el aeropuerto esperando tu vuelo y tomando un Chaha latte...
[4169] BCS _GEO I'm at Aeropuerto De La Paz in La Paz, BCS _URL
[35573] Chihuahua _GEO Nunca había visto tan aperrado (lleno de gente) el aeropuerto de Chihiuas
[31873] Mexico_City _GEO _USR que acciones vas a tomar contra esta ruta la 131 que corre de aeropuerto a toreo sobre el eje 1 norte., Pero que todos los días se sale de su ramal y ocasiona invasiones de pasos peatonales, cruce de semáforos _URL
[29411] Mexico _GEO _USR De que hablas yo vivo junto a miles de personas que tienen trabajo en el aeropuerto, de que hablas????
[15458] Mexico_City _GEO Se me acabo el #pinchi #corrido #bisnächstesmal #lasquierogurls _USR ✈️🤟🏼 en Aeropuerto Internacional de la Ciudad de México _URL
[11657] Puebla _GEO _USR posteo mi video! Wooww creo que es mi mayor logro en la vida! 😂🙈 Fuera de broma Noroñ